# 📄 OCR Comparison: GLM-OCR vs DeepSeek-OCR-2

So sánh hiệu quả OCR giữa **[GLM-OCR](https://huggingface.co/zai-org/GLM-OCR)** và **[DeepSeek-OCR-2](https://huggingface.co/deepseek-ai/DeepSeek-OCR-2)**.

**Cách dùng:** Chạy tất cả cell theo thứ tự từ trên xuống dưới.

> ⚠️ **Lưu ý VRAM:** Colab T4 chỉ có 16GB VRAM. Khi chọn "both", notebook sẽ chạy tuần tự từng model (load → OCR → unload).

## 1️⃣ Kiểm tra GPU & Cài đặt

In [ ]:
# === KIỂM TRA GPU ===
!nvidia-smi
import torch
print(f"\n✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("❌ Không có GPU! Vào Runtime → Change runtime type → chọn GPU")

In [ ]:
# === CÀI ĐẶT DEPENDENCIES ===
import sys
print(f"Python: {sys.executable}")

# --- Dependencies chung ---
!{sys.executable} -m pip install pymupdf openai -q
!apt-get install -y poppler-utils -qq 2>/dev/null

# --- GLM-OCR ---
!{sys.executable} -m pip install -U vllm --extra-index-url https://wheels.vllm.ai/nightly -q
!{sys.executable} -m pip install git+https://github.com/huggingface/transformers.git -q
![ ! -d /content/GLM-OCR ] && git clone https://github.com/zai-org/GLM-OCR.git /content/GLM-OCR
!cd /content/GLM-OCR && {sys.executable} -m pip install -e . -q

# --- DeepSeek-OCR-2 ---
![ ! -d /content/DeepSeek-OCR-2 ] && git clone https://github.com/deepseek-ai/DeepSeek-OCR-2.git /content/DeepSeek-OCR-2
!{sys.executable} -m pip install einops addict easydict -q
# flash-attn: thử cài pre-built, bỏ qua nếu lỗi (model vẫn chạy được)
!{sys.executable} -m pip install flash-attn --no-build-isolation -q 2>/dev/null || echo '⚠️ flash-attn không cài được, sẽ dùng eager attention'

import vllm
print(f"\n✅ vLLM version: {vllm.__version__}")
print("✅ Cài đặt hoàn tất!")

## 2️⃣ Upload PDF từ Google Drive

In [ ]:
import os
import shutil
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# ===== CẤU HÌNH ĐƯỜNG DẪN =====
DRIVE_PDF_DIR = "/content/drive/MyDrive/input-pdfs"  # ← Sửa nếu folder khác tên
INPUT_DIR = "/content/input_pdfs"
OUTPUT_DIR = "/content/output_markdown"
# ================================

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Copy tất cả file PDF từ Drive
pdf_count = 0
for f in sorted(os.listdir(DRIVE_PDF_DIR)):
    if f.lower().endswith('.pdf'):
        src = os.path.join(DRIVE_PDF_DIR, f)
        dst = os.path.join(INPUT_DIR, f)
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / (1024 * 1024)
        print(f"  ✅ {f} ({size_mb:.1f} MB)")
        pdf_count += 1

print(f"\n📄 Đã copy {pdf_count} file PDF từ Google Drive")
print(f"📂 Input:  {INPUT_DIR}")
print(f"📂 Output: {OUTPUT_DIR}")

## 3️⃣ Chọn Model OCR

In [ ]:
# ╔══════════════════════════════════════════════╗
# ║          CHỌN MODEL OCR ĐỂ SỬ DỤNG          ║
# ╠══════════════════════════════════════════════╣
# ║  "glm-ocr"       → Chỉ dùng GLM-OCR         ║
# ║  "deepseek-ocr2" → Chỉ dùng DeepSeek-OCR-2  ║
# ║  "both"          → Chạy cả 2 để so sánh     ║
# ╚══════════════════════════════════════════════╝

MODEL_CHOICE = "both"  # ← Đổi giá trị ở đây

DPI = 200
USE_SDK = True  # GLM-OCR: True = SDK, False = vLLM API

# --- Validation ---
assert MODEL_CHOICE in ("glm-ocr", "deepseek-ocr2", "both"), \
    f"❌ MODEL_CHOICE phải là 'glm-ocr', 'deepseek-ocr2', hoặc 'both'. Nhận: '{MODEL_CHOICE}'"

model_labels = {
    "glm-ocr": "🔵 GLM-OCR (0.9B)",
    "deepseek-ocr2": "🟢 DeepSeek-OCR-2 (3B MoE)",
    "both": "🔵 GLM-OCR + 🟢 DeepSeek-OCR-2 (so sánh)"
}
print(f"\n✅ Model: {model_labels[MODEL_CHOICE]}")
if MODEL_CHOICE == "both":
    print("   ⚠️ Sẽ chạy tuần tự: GLM-OCR → unload → DeepSeek-OCR-2")
    print("   ⏱️ Thời gian tổng sẽ gấp ~2x do phải load/unload model")

## 4️⃣ Khởi động & Chạy OCR

In [ ]:
import os
import sys
import gc
import glob
import time
import base64
import shutil
import subprocess
import traceback
import requests
import torch
import fitz  # PyMuPDF
from pathlib import Path
from openai import OpenAI

# ===== CẤU HÌNH =====
INPUT_DIR = "/content/input_pdfs"
OUTPUT_DIR = "/content/output_markdown"
VLLM_PORT = 8899
# =====================

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ══════════════════════════════════════════════
#  HELPER: PDF → Images
# ══════════════════════════════════════════════
def pdf_to_images(pdf_path, dpi=200):
    """Chuyển PDF thành list ảnh (1 ảnh/trang)."""
    doc = fitz.open(pdf_path)
    image_paths = []
    temp_dir = f"/tmp/pdf_pages/{Path(pdf_path).stem}"
    os.makedirs(temp_dir, exist_ok=True)
    for page_num in range(len(doc)):
        page = doc[page_num]
        zoom = dpi / 72
        pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
        img_path = os.path.join(temp_dir, f"page_{page_num+1:04d}.png")
        pix.save(img_path)
        image_paths.append(img_path)
    doc.close()
    return image_paths


# ══════════════════════════════════════════════
#  GLM-OCR: vLLM Server + OpenAI API
# ══════════════════════════════════════════════
vllm_process = None

def start_vllm_server():
    """Khởi động vLLM server cho GLM-OCR."""
    global vllm_process
    import subprocess, time, requests
    MODEL_NAME = "zai-org/GLM-OCR"
    os.system(f"fuser -k {VLLM_PORT}/tcp 2>/dev/null")
    time.sleep(2)
    print("🚀 Khởi động vLLM server (GLM-OCR)...")
    vllm_process = subprocess.Popen(
        [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
         "--model", MODEL_NAME, "--allowed-local-media-path", "/",
         "--port", str(VLLM_PORT), "--served-model-name", "glm-ocr",
         "--gpu-memory-utilization", "0.90", "--max-model-len", "16384"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print("⏳ Chờ server sẵn sàng...")
    start = time.time()
    while time.time() - start < 600:
        try:
            if requests.get(f"http://localhost:{VLLM_PORT}/health", timeout=2).status_code == 200:
                print(f"✅ vLLM server sẵn sàng! ({int(time.time()-start)}s)")
                return True
        except: pass
        if vllm_process.poll() is not None:
            out, _ = vllm_process.communicate()
            print(f"❌ vLLM dừng bất ngờ!\n{out[-2000:]}")
            return False
        e = int(time.time()-start)
        if e % 30 == 0 and e > 0: print(f"   ...{e}s")
        time.sleep(5)
    print("❌ Timeout")
    return False

def stop_vllm_server():
    """Dừng vLLM server và giải phóng VRAM."""
    global vllm_process
    if vllm_process:
        try:
            vllm_process.terminate()
            vllm_process.wait(timeout=10)
        except:
            os.system(f"fuser -k {VLLM_PORT}/tcp 2>/dev/null")
        vllm_process = None
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(3)
    print("🧹 GLM-OCR server đã dừng, VRAM đã giải phóng")

def ocr_with_glm_sdk(image_paths):
    """OCR bằng GLM-OCR SDK."""
    from glmocr import parse
    return parse(image_paths)

def ocr_with_glm_api(image_paths):
    """OCR bằng GLM-OCR qua OpenAI API."""
    client = OpenAI(api_key="EMPTY", base_url=f"http://localhost:{VLLM_PORT}/v1")
    all_md = []
    page_times = []
    for i, img_path in enumerate(image_paths):
        t_page = time.time()
        with open(img_path, "rb") as f:
            img_b64 = base64.b64encode(f.read()).decode("utf-8")
        resp = client.chat.completions.create(
            model="glm-ocr",
            messages=[{"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}},
                {"type": "text", "text": "Text Recognition:"},
            ]}],
            max_tokens=8192, temperature=0.01,
        )
        all_md.append(resp.choices[0].message.content)
        pt = time.time() - t_page
        page_times.append(pt)
        print(f"      Trang {i+1}/{len(image_paths)} ✓ ({pt:.1f}s)")
    return "\n\n---\n\n".join(all_md), page_times


# ══════════════════════════════════════════════
#  DeepSeek-OCR-2: Transformers Direct Inference
# ══════════════════════════════════════════════
deepseek_model = None
deepseek_tokenizer = None

def load_deepseek_model():
    """Load DeepSeek-OCR-2 vào GPU."""
    global deepseek_model, deepseek_tokenizer
    from transformers import AutoModel, AutoTokenizer
    print("🚀 Đang tải DeepSeek-OCR-2 (lần đầu sẽ download ~6GB)...")
    t0 = time.time()
    model_name = "deepseek-ai/DeepSeek-OCR-2"
    deepseek_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    try:
        import flash_attn
        attn_impl = 'flash_attention_2'
        print('   ✅ Dùng Flash Attention 2')
    except ImportError:
        attn_impl = 'eager'
        print('   ⚠️ flash-attn chưa cài, dùng eager attention')
    deepseek_model = AutoModel.from_pretrained(
        model_name,
        _attn_implementation=attn_impl,
        trust_remote_code=True,
        use_safetensors=True
    )
    deepseek_model = deepseek_model.eval().cuda().to(torch.bfloat16)
    print(f"✅ DeepSeek-OCR-2 đã tải! ({time.time()-t0:.0f}s)")

def unload_deepseek_model():
    """Unload DeepSeek-OCR-2 khỏi GPU."""
    global deepseek_model, deepseek_tokenizer
    del deepseek_model, deepseek_tokenizer
    deepseek_model = None
    deepseek_tokenizer = None
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(3)
    print("🧹 DeepSeek-OCR-2 đã unload, VRAM đã giải phóng")

def ocr_with_deepseek(image_paths):
    """OCR bằng DeepSeek-OCR-2 via Transformers."""
    all_md = []
    page_times = []
    prompt = '<image>\n<|grounding|>Convert the document to markdown.'
    for i, img_path in enumerate(image_paths):
        t_page = time.time()
        out_dir = f'/tmp/deepseek_ocr_out/page_{i}'
        os.makedirs(out_dir, exist_ok=True)
        deepseek_model.infer(
            deepseek_tokenizer,
            prompt=prompt,
            image_file=img_path,
            output_path=out_dir,
            base_size=1024,
            image_size=768,
            crop_mode=True,
            save_results=True
        )
        # Đọc kết quả từ file đã lưu
        result_file = os.path.join(out_dir, 'result.mmd')
        if os.path.exists(result_file):
            with open(result_file, 'r', encoding='utf-8') as f:
                text = f.read().strip()
        else:
            text = '[Không có kết quả]'
        all_md.append(text)
        pt = time.time() - t_page
        page_times.append(pt)
        print(f"      Trang {i+1}/{len(image_paths)} ✓ ({pt:.1f}s)")
    # Cleanup temp
    shutil.rmtree('/tmp/deepseek_ocr_out', ignore_errors=True)
    return "\n\n---\n\n".join(all_md), page_times

# ══════════════════════════════════════════════
#  PROCESS: Xử lý 1 PDF với 1 model
# ══════════════════════════════════════════════
def process_pdf_single(pdf_path, model_name, images=None):
    """Xử lý 1 PDF với 1 model cụ thể. Trả về dict kết quả."""
    pdf_name = Path(pdf_path).stem
    suffix = "_glm" if model_name == "glm-ocr" else "_deepseek"
    label = "🔵 GLM-OCR" if model_name == "glm-ocr" else "🟢 DeepSeek-OCR-2"

    # PDF → ảnh (chỉ convert 1 lần, có thể reuse)
    t_img = time.time()
    if images is None:
        print(f"   📸 PDF → ảnh (DPI={DPI})...", end=" ")
        images = pdf_to_images(pdf_path, dpi=DPI)
        t_img_elapsed = time.time() - t_img
        print(f"{len(images)} trang ({t_img_elapsed:.1f}s)")
    else:
        t_img_elapsed = 0
        print(f"   📸 Dùng lại {len(images)} ảnh đã convert")

    # OCR
    t_ocr = time.time()
    print(f"   🔍 [{label}] OCR {len(images)} trang...")
    md_text = ""
    page_times = []
    try:
        if model_name == "glm-ocr":
            if USE_SDK:
                try:
                    result = ocr_with_glm_sdk(images)
                    out_dir = os.path.join(OUTPUT_DIR, pdf_name)
                    os.makedirs(out_dir, exist_ok=True)
                    result.save(output_dir=out_dir)
                    md_file = os.path.join(out_dir, "result.md")
                    md_text = open(md_file, 'r', encoding='utf-8').read() if os.path.exists(md_file) else str(result)
                    page_times = [0] * len(images)  # SDK không track per-page
                except Exception as e:
                    print(f"   ⚠️ SDK lỗi, dùng API: {e}")
                    md_text, page_times = ocr_with_glm_api(images)
            else:
                md_text, page_times = ocr_with_glm_api(images)
        elif model_name == "deepseek-ocr2":
            md_text, page_times = ocr_with_deepseek(images)
    except Exception as e:
        print(f"   ❌ Lỗi: {e}")
        traceback.print_exc()
        return {"file": Path(pdf_path).name, "pages": len(images),
                "model": model_name, "time_img": round(t_img_elapsed,2),
                "time_ocr": 0, "time_total": 0, "page_times": [],
                "status": f"❌ {str(e)[:50]}"}

    t_ocr_elapsed = time.time() - t_ocr
    t_total = t_img_elapsed + t_ocr_elapsed

    # Lưu output
    out_suffix = suffix if MODEL_CHOICE == "both" else ""
    md_out = os.path.join(OUTPUT_DIR, f"{pdf_name}{out_suffix}.md")
    model_label_short = "GLM-OCR" if model_name == "glm-ocr" else "DeepSeek-OCR-2"
    with open(md_out, 'w', encoding='utf-8') as f:
        f.write(f"<!-- OCR by {model_label_short} | Source: {Path(pdf_path).name} -->\n\n{md_text}")

    pp_speed = len(images) / max(t_ocr_elapsed, 0.1)
    print(f"   ✅ [{label}] Xong: {t_ocr_elapsed:.1f}s OCR ({pp_speed:.2f} trang/s) | 💾 {Path(md_out).name}")

    return {
        "file": Path(pdf_path).name,
        "pages": len(images),
        "model": model_name,
        "time_img": round(t_img_elapsed, 2),
        "time_ocr": round(t_ocr_elapsed, 2),
        "time_total": round(t_total, 2),
        "page_times": [round(pt, 2) for pt in page_times],
        "avg_per_page": round(t_ocr_elapsed / max(len(images), 1), 2),
        "pages_per_sec": round(pp_speed, 2),
        "status": "✅"
    }


# ══════════════════════════════════════════════
#  BÁO CÁO SO SÁNH
# ══════════════════════════════════════════════
def print_comparison_report(all_results, t_total_all):
    """In bảng so sánh hiệu suất giữa các model."""
    models_used = sorted(set(r["model"] for r in all_results))
    files = sorted(set(r["file"] for r in all_results))

    # Header
    print(f"\n{'═'*75}")
    print(f"{'📊 BÁO CÁO SO SÁNH OCR':^75}")
    print(f"{'═'*75}")

    if len(models_used) == 1:
        # Báo cáo đơn model
        model = models_used[0]
        label = "🔵 GLM-OCR" if model == "glm-ocr" else "🟢 DeepSeek-OCR-2"
        results = [r for r in all_results if r["model"] == model]

        print(f"\n Model: {label}")
        print(f" {'─'*70}")
        print(f" {'File':<30} {'Trang':>6} {'PDF→Ảnh':>9} {'OCR':>9} {'TB/trang':>10} {'Trạng thái':>10}")
        print(f" {'─'*70}")
        for r in results:
            print(f" {r['file']:<30} {r['pages']:>6} {r['time_img']:>8.1f}s {r['time_ocr']:>8.1f}s {r.get('avg_per_page',0):>9.2f}s {r['status']:>10}")
        print(f" {'─'*70}")

        ok_results = [r for r in results if '✅' in r['status']]
        total_pages = sum(r['pages'] for r in ok_results)
        total_ocr = sum(r['time_ocr'] for r in ok_results)
        print(f" {'TỔNG':<30} {total_pages:>6} {'':>9} {total_ocr:>8.1f}s {total_ocr/max(total_pages,1):>9.2f}s {len(ok_results)}/{len(results)} OK")
        if total_pages > 0:
            print(f"\n ⚡ Tốc độ OCR: {total_pages/total_ocr:.2f} trang/giây")

    else:
        # Báo cáo so sánh 2 model
        print(f"\n {'File':<25} {'Trang':>5}  │ {'🔵 GLM-OCR':>15} {'(TB/p)':>8}  │ {'🟢 DeepSeek-2':>15} {'(TB/p)':>8}  │ {'Nhanh hơn':>12}")
        print(f" {'─'*110}")

        glm_total_ocr = 0
        ds_total_ocr = 0
        total_pages = 0

        for fname in files:
            glm_r = next((r for r in all_results if r['file']==fname and r['model']=='glm-ocr'), None)
            ds_r = next((r for r in all_results if r['file']==fname and r['model']=='deepseek-ocr2'), None)

            pages = (glm_r or ds_r)['pages']
            total_pages += pages

            glm_str = f"{glm_r['time_ocr']:.1f}s" if glm_r and '✅' in glm_r['status'] else "—"
            glm_pp = f"{glm_r['avg_per_page']:.2f}s" if glm_r and '✅' in glm_r['status'] else "—"
            ds_str = f"{ds_r['time_ocr']:.1f}s" if ds_r and '✅' in ds_r['status'] else "—"
            ds_pp = f"{ds_r['avg_per_page']:.2f}s" if ds_r and '✅' in ds_r['status'] else "—"

            if glm_r and '✅' in glm_r['status']: glm_total_ocr += glm_r['time_ocr']
            if ds_r and '✅' in ds_r['status']: ds_total_ocr += ds_r['time_ocr']

            # So sánh
            winner = "—"
            if glm_r and ds_r and '✅' in glm_r['status'] and '✅' in ds_r['status']:
                if glm_r['time_ocr'] < ds_r['time_ocr']:
                    pct = (ds_r['time_ocr']-glm_r['time_ocr'])/ds_r['time_ocr']*100
                    winner = f"🔵 +{pct:.0f}%"
                elif ds_r['time_ocr'] < glm_r['time_ocr']:
                    pct = (glm_r['time_ocr']-ds_r['time_ocr'])/glm_r['time_ocr']*100
                    winner = f"🟢 +{pct:.0f}%"
                else:
                    winner = "⚖️ Bằng"

            print(f" {fname:<25} {pages:>5}  │ {glm_str:>15} {glm_pp:>8}  │ {ds_str:>15} {ds_pp:>8}  │ {winner:>12}")

        print(f" {'─'*110}")

        # Tổng kết
        glm_speed = total_pages/max(glm_total_ocr,0.1) if glm_total_ocr > 0 else 0
        ds_speed = total_pages/max(ds_total_ocr,0.1) if ds_total_ocr > 0 else 0

        print(f" {'TỔNG':<25} {total_pages:>5}  │ {glm_total_ocr:>14.1f}s {'':>8}  │ {ds_total_ocr:>14.1f}s {'':>8}  │")
        print(f" {'Tốc độ TB':<25} {'':>5}  │ {glm_speed:>12.2f} p/s {'':>8}  │ {ds_speed:>12.2f} p/s {'':>8}  │")

        # Kết luận
        print(f"\n {'─'*50}")
        if glm_total_ocr > 0 and ds_total_ocr > 0:
            if glm_total_ocr < ds_total_ocr:
                pct = (ds_total_ocr - glm_total_ocr) / ds_total_ocr * 100
                print(f" 🏆 Tổng kết: 🔵 GLM-OCR nhanh hơn {pct:.1f}%")
            elif ds_total_ocr < glm_total_ocr:
                pct = (glm_total_ocr - ds_total_ocr) / glm_total_ocr * 100
                print(f" 🏆 Tổng kết: 🟢 DeepSeek-OCR-2 nhanh hơn {pct:.1f}%")
            else:
                print(f" 🏆 Tổng kết: Hai model ngang tốc độ")

    print(f"\n ⏱️  Tổng thời gian chạy: {t_total_all:.1f}s")
    print(f" 📂 Kết quả: {OUTPUT_DIR}")
    print(f"{'═'*75}")


# ══════════════════════════════════════════════
#  MAIN: Tìm PDF và chạy OCR
# ══════════════════════════════════════════════
pdf_files = sorted(set(
    glob.glob(os.path.join(INPUT_DIR, "*.pdf")) +
    glob.glob(os.path.join(INPUT_DIR, "**/*.pdf"), recursive=True)
))

if not pdf_files:
    print(f"❌ Không tìm thấy file PDF nào trong {INPUT_DIR}")
else:
    print(f"📋 {len(pdf_files)} file PDF:")
    for i, f in enumerate(pdf_files, 1):
        print(f"   {i}. {Path(f).name} ({os.path.getsize(f)/1024/1024:.1f} MB)")

    all_results = []
    t_grand_start = time.time()

    models_to_run = []
    if MODEL_CHOICE in ("glm-ocr", "both"):
        models_to_run.append("glm-ocr")
    if MODEL_CHOICE in ("deepseek-ocr2", "both"):
        models_to_run.append("deepseek-ocr2")

    # Cache ảnh đã convert để không phải convert lại
    images_cache = {}

    for model_name in models_to_run:
        label = "🔵 GLM-OCR" if model_name == "glm-ocr" else "🟢 DeepSeek-OCR-2"
        print(f"\n{'▓'*60}")
        print(f"  {label} — Bắt đầu xử lý {len(pdf_files)} file")
        print(f"{'▓'*60}")

        # Load model
        t_load = time.time()
        if model_name == "glm-ocr":
            if not start_vllm_server():
                print("❌ Không thể khởi động GLM-OCR, bỏ qua")
                continue
        elif model_name == "deepseek-ocr2":
            load_deepseek_model()
        load_time = time.time() - t_load
        print(f"   ⏱️ Thời gian load model: {load_time:.0f}s")

        # Xử lý từng file
        for idx, pdf in enumerate(pdf_files, 1):
            print(f"\n[{idx}/{len(pdf_files)}]", end="")
            print(f"\n{'='*60}")
            print(f"📄 {Path(pdf).name}")

            # Convert ảnh (cache lại)
            pdf_key = pdf
            if pdf_key not in images_cache:
                images_cache[pdf_key] = pdf_to_images(pdf, dpi=DPI)
                print(f"   📸 Đã convert {len(images_cache[pdf_key])} trang (DPI={DPI})")

            try:
                result = process_pdf_single(pdf, model_name, images=images_cache[pdf_key])
                all_results.append(result)
            except Exception as e:
                print(f"   ❌ {e}")
                traceback.print_exc()
                all_results.append({
                    "file": Path(pdf).name, "pages": 0, "model": model_name,
                    "time_img": 0, "time_ocr": 0, "time_total": 0,
                    "page_times": [], "avg_per_page": 0, "pages_per_sec": 0,
                    "status": f"❌ {str(e)[:50]}"
                })

        # Unload model (nếu cần chạy model tiếp theo)
        if MODEL_CHOICE == "both":
            if model_name == "glm-ocr":
                stop_vllm_server()
            elif model_name == "deepseek-ocr2":
                unload_deepseek_model()

    t_grand_total = time.time() - t_grand_start

    # Dọn dẹp images cache
    for pdf_key in images_cache:
        stem = Path(pdf_key).stem
        shutil.rmtree(f"/tmp/pdf_pages/{stem}", ignore_errors=True)

    # In báo cáo
    print_comparison_report(all_results, t_grand_total)

## 5️⃣ Xem kết quả

In [ ]:
import os, glob
from pathlib import Path
from IPython.display import Markdown, display

OUTPUT_DIR = "/content/output_markdown"
md_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.md")))

print(f"📄 {len(md_files)} file markdown:\n")
for i, f in enumerate(md_files):
    print(f"  [{i}] {Path(f).name} ({os.path.getsize(f)/1024:.1f} KB)")

# === Xem 1 file (đổi số để xem file khác) ===
FILE_INDEX = 0

if md_files:
    print(f"\n{'='*60}")
    print(f"📄 Đang xem: [{FILE_INDEX}] {Path(md_files[FILE_INDEX]).name}\n")
    with open(md_files[FILE_INDEX], 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))

## 6️⃣ Tải kết quả

In [ ]:
import shutil
from google.colab import files

OUTPUT_DIR = "/content/output_markdown"

# Nén và tải về
shutil.make_archive("/content/ocr_results", 'zip', OUTPUT_DIR)
print("📦 Đã nén kết quả")
files.download("/content/ocr_results.zip")

# Hoặc lưu vào Google Drive:
# shutil.copytree(OUTPUT_DIR, "/content/drive/MyDrive/OCR_Results", dirs_exist_ok=True)

## 🧹 Dọn dẹp

In [ ]:
import shutil, gc, torch

# Dừng vLLM server
try:
    vllm_process.terminate()
    vllm_process.wait(timeout=10)
    print("✅ Đã dừng vLLM server")
except:
    !fuser -k 8899/tcp 2>/dev/null
    print("⚠️ Đã force kill vLLM server")

# Unload DeepSeek model nếu còn
try:
    if deepseek_model is not None:
        del deepseek_model, deepseek_tokenizer
        print("✅ Đã unload DeepSeek-OCR-2")
except: pass

gc.collect()
torch.cuda.empty_cache()
shutil.rmtree("/tmp/pdf_pages", ignore_errors=True)
print("🗑️ Đã dọn dẹp")